# YOLOv8 PPE Detection - Simple Training

이 노트북은 사전 추출된 데이터셋으로 YOLOv8을 학습합니다.

## 1. Install Libraries

In [1]:
!pip install -q ultralytics opencv-python pillow matplotlib pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.5 MB/s eta 0:00:00


## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Setup Paths and Check Data

In [3]:
import os
import glob

# Path setup
data_path = '/content/drive/MyDrive/Colab_Notebooks/data'
output_path = '/content/drive/MyDrive/Colab_Notebooks/results'

os.makedirs(output_path, exist_ok=True)

print("=" * 60)
print("📂 DATA STRUCTURE CHECK")
print("=" * 60)

# Check directory structure
print(f"\n✓ Data folder: {data_path}")
print(f"✓ Output folder: {output_path}\n")

# List all folders and image counts
for root, dirs, files in os.walk(data_path):
    rel_path = os.path.relpath(root, data_path)
    if rel_path == '.':
        rel_path = '[root]'

    # Count files by type
    imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    labels = [f for f in files if f.endswith('.txt')]
    yamls = [f for f in files if f.endswith(('.yaml', '.yml'))]

    if imgs:
        print(f"📷 {rel_path}: {len(imgs)} images")
    if labels:
        print(f"📝 {rel_path}: {len(labels)} labels")
    if yamls:
        print(f"⚙️  {rel_path}: {yamls}")

print("\n" + "=" * 60)

📂 DATA STRUCTURE CHECK

✓ Data folder: /content/drive/MyDrive/Colab_Notebooks/data
✓ Output folder: /content/drive/MyDrive/Colab_Notebooks/results

📝 [root]: 2 labels
⚙️  [root]: ['data.yaml']
📷 train/images: 2378 images



## 4. Prepare data.yaml

In [ ]:
import yaml
import glob

# Start with basic configuration
data_yaml = {
    'path': data_path,
    'nc': 4,
    'names': ['0', '1', '2', 'mask']
}

print("=" * 60)
print("📝 CREATING data.yaml")
print("=" * 60)

# Check TRAIN (required)
train_candidates = [
    os.path.join(data_path, 'train/images'),
    os.path.join(data_path, 'train'),
]
train_path = None
for path in train_candidates:
    if os.path.exists(path):
        # Check if it has images
        imgs = glob.glob(os.path.join(path, '*.jpg')) + glob.glob(os.path.join(path, '*.png'))
        if imgs:
            train_path = path
            data_yaml['train'] = path
            print(f"✓ TRAIN:  {path} ({len(imgs)} images)")
            break

if not train_path:
    raise FileNotFoundError(f"❌ No train/images folder found in {data_path}")

# Check VAL (required by YOLOv8)
val_candidates = [
    os.path.join(data_path, 'valid/images'),
    os.path.join(data_path, 'val/images'),
    os.path.join(data_path, 'valid'),
    os.path.join(data_path, 'val'),
]
val_path = None
for path in val_candidates:
    if os.path.exists(path):
        imgs = glob.glob(os.path.join(path, '*.jpg')) + glob.glob(os.path.join(path, '*.png'))
        if imgs:
            val_path = path
            data_yaml['val'] = path
            print(f"✓ VAL:    {path} ({len(imgs)} images)")
            break

if not val_path:
    # Use train as val if val not found
    data_yaml['val'] = train_path
    print(f"⚠ VAL:    Not found - using train path for validation")
    print(f"         {train_path}")

# Check TEST (optional)
test_candidates = [
    os.path.join(data_path, 'test/images'),
    os.path.join(data_path, 'test'),
]
test_path = None
for path in test_candidates:
    if os.path.exists(path):
        imgs = glob.glob(os.path.join(path, '*.jpg')) + glob.glob(os.path.join(path, '*.png'))
        if imgs:
            test_path = path
            data_yaml['test'] = path
            print(f"✓ TEST:   {path} ({len(imgs)} images)")
            break

if not test_path:
    print(f"⚠ TEST:   Not found (optional)")

# Save data.yaml
yaml_path = os.path.join(data_path, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"\n✓ Saved: {yaml_path}")
print("=" * 60)

📝 CREATING data.yaml
✓ TRAIN:  /content/drive/MyDrive/Colab_Notebooks/data/train/images (4768 images)
⚠ VAL:    Not found (will use train for validation)
⚠ TEST:   Not found (optional)

✓ Saved: /content/drive/MyDrive/Colab_Notebooks/data/data.yaml


## 5. Train YOLOv8 Model

In [9]:
from ultralytics import YOLO
import torch

# Select device automatically
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load model
model = YOLO('yolov8m.pt')

# Train
print("=" * 60)
print("🚀 START TRAINING")
print("=" * 60)

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=device,
    project=output_path,
    name='ppe_detection',
    verbose=True
)

print("\n" + "=" * 60)
print("✓ TRAINING COMPLETED")
print("=" * 60)

Using device: 0
🚀 START TRAINING
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab_Notebooks/data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe_detection4, nbs=64, nms=False, opset=None, optimize=Fal

RuntimeError: Dataset '/content/drive/MyDrive/Colab_Notebooks/data/data.yaml' error ❌ /content/drive/MyDrive/Colab_Notebooks/data/data.yaml 'val:' key missing ❌.
'train' and 'val' are required in all data YAMLs.

## 6. View Results

In [6]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Find training results
train_dir = os.path.join(output_path, 'ppe_detection')
metric_files = glob.glob(os.path.join(train_dir, '*.png'))

if metric_files:
    print(f"Found {len(metric_files)} metric images\n")

    # Display first 4 metrics
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()

    for idx, img_path in enumerate(metric_files[:4]):
        try:
            img = mpimg.imread(img_path)
            axes[idx].imshow(img)
            axes[idx].set_title(os.path.basename(img_path), fontsize=10)
            axes[idx].axis('off')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

    plt.tight_layout()
    plt.show()

    print(f"\n✓ Results saved to: {train_dir}")
else:
    print(f"No metric images found in {train_dir}")

No metric images found in /content/drive/MyDrive/Colab_Notebooks/results/ppe_detection
